# Churn, Retention and Cohort Analysis V1

This notebook focuses on the core subscription questions for the premium budgeting app:
- which customer groups churn the most?
- how does paid retention evolve over time?
- which segments appear most at risk?
- which business actions could plausibly improve retention?

This is a V1 business analysis notebook:
- no machine learning
- no advanced survival modelling
- simple and explicit cohort logic
- clear approximations when the simulated dataset structure requires them

## 1. Setup

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

plt.style.use('seaborn-v0_8-whitegrid')
pd.set_option('display.max_columns', 100)
pd.set_option('display.float_format', lambda value: f'{value:,.3f}')

DATA_DIR = Path('../data/raw')
DATASET_END = pd.Timestamp('2025-12-31')
MONTHS = pd.date_range('2025-01-01', '2025-12-01', freq='MS')

## 2. Load Data

In [ ]:
customers = pd.read_csv(DATA_DIR / 'customers.csv', parse_dates=['signup_date'])
subscriptions = pd.read_csv(
    DATA_DIR / 'subscriptions.csv',
    parse_dates=['trial_start_date', 'trial_end_date', 'subscription_start_date', 'subscription_end_date'],
)
payments = pd.read_csv(DATA_DIR / 'payments.csv', parse_dates=['payment_date'])
product_events = pd.read_csv(DATA_DIR / 'product_events.csv', parse_dates=['event_date'])
monthly_customer_activity = pd.read_csv(
    DATA_DIR / 'monthly_customer_activity.csv',
    parse_dates=['activity_month'],
)

print(f'customers: {customers.shape[0]:,}')
print(f'subscriptions: {subscriptions.shape[0]:,}')
print(f'payments: {payments.shape[0]:,}')
print(f'product_events: {product_events.shape[0]:,}')
print(f'monthly_customer_activity: {monthly_customer_activity.shape[0]:,}')

## 3. Definitions Used in This Notebook

Definitions kept intentionally simple and explicit:

- **Converted customer**: `converted_to_paid = True`
- **Cancelled customer**: a converted customer with `subscription_status = 'cancelled'`
- **Global churn share**: cancelled paid customers / converted paid customers
- **Monthly churn rate**: customers cancelled during month M / customers active at the start of month M
- **Paid retention**: share of converted customers still active after N months since conversion
- **Usage segment**: latest observed usage segment from `monthly_customer_activity`

Important limitation:
- `monthly_customer_activity` contains observed months with events only
- it does not materialize explicit zero-activity months
- usage-based churn comparisons are therefore useful as directional signals, not as a final causal proof

## 4. Analysis Base Tables

In [ ]:
activation_events = product_events.merge(customers[['customer_id', 'signup_date']], on='customer_id', how='left')
activation_events = activation_events[
    activation_events['event_date'] <= activation_events['signup_date'] + pd.Timedelta(days=6)
]

activation_status = (
    activation_events.groupby('customer_id')['event_type']
    .agg(list)
    .apply(
        lambda events: (
            any(event in {'bank_account_connected', 'transaction_imported'} for event in events)
            and 'budget_created' in events
        )
    )
    .rename('is_activated')
)

customer_base = customers.merge(activation_status, on='customer_id', how='left')
customer_base['is_activated'] = customer_base['is_activated'].eq(True)

paid_customers = subscriptions[subscriptions['converted_to_paid']].copy()
paid_customers = paid_customers.merge(
    customer_base[['customer_id', 'acquisition_channel', 'is_activated']],
    on='customer_id',
    how='left',
)
paid_customers['conversion_month'] = paid_customers['subscription_start_date'].values.astype('datetime64[M]')
paid_customers['is_cancelled'] = paid_customers['subscription_status'].eq('cancelled')

latest_usage = (
    monthly_customer_activity.sort_values(['customer_id', 'activity_month'])
    .groupby('customer_id')
    .tail(1)[['customer_id', 'usage_segment', 'activity_month']]
    .rename(columns={'activity_month': 'latest_activity_month'})
)

paid_customers = paid_customers.merge(latest_usage, on='customer_id', how='left')
paid_customers['usage_segment'] = paid_customers['usage_segment'].fillna('no_observed_activity')

observed_usage = (
    monthly_customer_activity.groupby('customer_id')
    .agg(
        observed_months=('activity_month', 'nunique'),
        avg_app_opens=('nb_app_opens', 'mean'),
        avg_transactions=('nb_transactions_imported', 'mean'),
        avg_dashboard_views=('nb_dashboard_views', 'mean'),
    )
    .reset_index()
)

paid_customers = paid_customers.merge(observed_usage, on='customer_id', how='left')
paid_customers['avg_app_opens'] = paid_customers['avg_app_opens'].fillna(0.0)
paid_customers['avg_transactions'] = paid_customers['avg_transactions'].fillna(0.0)
paid_customers['avg_dashboard_views'] = paid_customers['avg_dashboard_views'].fillna(0.0)
paid_customers['observed_months'] = paid_customers['observed_months'].fillna(0).astype(int)

paid_customers.head()

## 5. Global Churn and Churn by Plan

In [ ]:
global_churn_share = paid_customers['is_cancelled'].mean()
churn_by_plan = (
    paid_customers.groupby('plan_type')['is_cancelled']
    .mean()
    .sort_values(ascending=False)
    .rename('churn_share')
    .to_frame()
)

pd.DataFrame([
    {'kpi': 'Global churn share among paid customers', 'value': global_churn_share}
])

In [ ]:
churn_by_plan

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
churn_by_plan['churn_share'].plot(kind='bar', ax=ax, color=['#dc2626', '#0f766e'])
ax.set_title('Churn Share by Plan')
ax.set_xlabel('Plan type')
ax.set_ylabel('Churn share')
plt.tight_layout()
plt.show()

**Observation**

Monthly subscriptions churn much more than annual subscriptions. This is expected and reinforces the strategic value of moving qualified customers toward longer commitments.

## 6. Churn by Acquisition Channel and Activation

In [ ]:
churn_by_channel = (
    paid_customers.groupby('acquisition_channel')['is_cancelled']
    .mean()
    .sort_values(ascending=False)
    .rename('churn_share')
    .to_frame()
)

churn_by_activation = (
    paid_customers.groupby('is_activated')['is_cancelled']
    .mean()
    .rename('churn_share')
    .to_frame()
)

display(churn_by_channel)
display(churn_by_activation)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

churn_by_channel['churn_share'].sort_values().plot(kind='barh', ax=axes[0], color='#2563eb')
axes[0].set_title('Churn Share by Acquisition Channel')
axes[0].set_xlabel('Churn share')

churn_by_activation['churn_share'].rename(index={False: 'Not activated', True: 'Activated'}).plot(
    kind='bar', ax=axes[1], color=['#f97316', '#16a34a']
)
axes[1].set_title('Churn Share: Activated vs Not Activated')
axes[1].set_xlabel('Activation status')
axes[1].set_ylabel('Churn share')

plt.tight_layout()
plt.show()

**Observation**

The same channels that convert better also tend to retain better. Activation remains a strong risk discriminator even after conversion, which makes onboarding quality a retention topic, not only a conversion topic.

## 7. Churn by Usage Segment

In [ ]:
churn_by_usage = (
    paid_customers.groupby('usage_segment')['is_cancelled']
    .mean()
    .sort_values(ascending=False)
    .rename('churn_share')
    .to_frame()
)
churn_by_usage

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
churn_by_usage['churn_share'].sort_values().plot(kind='barh', ax=ax, color='#8b5cf6')
ax.set_title('Churn Share by Latest Observed Usage Segment')
ax.set_xlabel('Churn share')
plt.tight_layout()
plt.show()

**Observation**

Lower observed usage is associated with much higher churn. This should be interpreted as a meaningful risk signal, while keeping in mind that zero-activity months are not explicitly present in the simulated monthly activity table.

## 8. Monthly Churn Evolution

Monthly churn rate is approximated here as:
- cancelled paid customers during month M
- divided by paid customers active at the start of month M

In [ ]:
monthly_churn_rows = []
for month_start in MONTHS:
    month_end = month_start + pd.offsets.MonthEnd(1)
    active_at_start = paid_customers[
        (paid_customers['subscription_start_date'] < month_start)
        & (
            paid_customers['subscription_end_date'].isna()
            | (paid_customers['subscription_end_date'] >= month_start)
        )
    ]
    cancelled_in_month = paid_customers[
        paid_customers['subscription_end_date'].notna()
        & (paid_customers['subscription_end_date'] >= month_start)
        & (paid_customers['subscription_end_date'] <= month_end)
    ]
    monthly_churn_rows.append(
        {
            'month': month_start,
            'active_paid_at_start': active_at_start['customer_id'].nunique(),
            'cancelled_in_month': cancelled_in_month['customer_id'].nunique(),
            'monthly_churn_rate': (
                cancelled_in_month['customer_id'].nunique() / active_at_start['customer_id'].nunique()
                if active_at_start['customer_id'].nunique() > 0 else 0.0
            ),
        }
    )

monthly_churn = pd.DataFrame(monthly_churn_rows)
monthly_churn

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
monthly_churn.plot(x='month', y='monthly_churn_rate', marker='o', ax=ax, color='#dc2626')
ax.set_title('Approximate Monthly Churn Rate')
ax.set_xlabel('Month')
ax.set_ylabel('Monthly churn rate')
plt.tight_layout()
plt.show()

**Observation**

The monthly churn line helps frame subscription risk over time. In this V1, it should be read as a practical business approximation rather than a billing-grade finance metric.

## 9. Paid Retention

In [ ]:
retention_rows = []
for months_since_conversion in range(0, 7):
    retained = paid_customers[
        paid_customers['subscription_end_date'].isna()
        | (
            paid_customers['subscription_end_date']
            > (paid_customers['subscription_start_date'] + pd.DateOffset(months=months_since_conversion) + pd.offsets.MonthEnd(0))
        )
    ]
    retention_rows.append(
        {
            'months_since_conversion': months_since_conversion,
            'paid_retention_rate': retained['customer_id'].nunique() / paid_customers['customer_id'].nunique(),
        }
    )

paid_retention = pd.DataFrame(retention_rows)
paid_retention

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
paid_retention.plot(x='months_since_conversion', y='paid_retention_rate', marker='o', ax=ax, color='#0f766e')
ax.set_title('Paid Retention Curve')
ax.set_xlabel('Months since conversion')
ax.set_ylabel('Retention rate')
plt.tight_layout()
plt.show()

**Observation**

Paid retention gives a more intuitive view than churn alone. It helps show how much of the paid base survives beyond the first months after conversion.

## 10. Converted Customer Cohorts

Simple cohort logic used here:
- cohorts are defined by `subscription_start_date` month
- for each cohort, we measure whether customers are still active after N months

In [ ]:
cohort_records = []
for cohort_month, cohort_frame in paid_customers.groupby('conversion_month'):
    cohort_size = cohort_frame['customer_id'].nunique()
    max_offset = int((DATASET_END.to_period('M') - pd.Timestamp(cohort_month).to_period('M')).n)
    for month_offset in range(0, max_offset + 1):
        cutoff = pd.Timestamp(cohort_month) + pd.DateOffset(months=month_offset) + pd.offsets.MonthEnd(0)
        retained_count = cohort_frame[
            cohort_frame['subscription_end_date'].isna()
            | (cohort_frame['subscription_end_date'] > cutoff)
        ]['customer_id'].nunique()
        cohort_records.append(
            {
                'cohort_month': pd.Timestamp(cohort_month),
                'month_offset': month_offset,
                'cohort_size': cohort_size,
                'retained_count': retained_count,
                'retention_rate': retained_count / cohort_size if cohort_size else 0.0,
            }
        )

cohort_table = pd.DataFrame(cohort_records)
cohort_pivot = cohort_table.pivot(index='cohort_month', columns='month_offset', values='retention_rate').sort_index()
cohort_pivot

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
heatmap = ax.imshow(cohort_pivot.fillna(pd.NA).astype(float), aspect='auto', cmap='YlGnBu', vmin=0, vmax=1)
ax.set_title('Paid Retention Cohort Heatmap')
ax.set_xlabel('Months since conversion')
ax.set_ylabel('Conversion cohort month')
ax.set_xticks(range(len(cohort_pivot.columns)))
ax.set_xticklabels(cohort_pivot.columns)
ax.set_yticks(range(len(cohort_pivot.index)))
ax.set_yticklabels([date.strftime('%Y-%m') for date in cohort_pivot.index])

for row_idx, row_label in enumerate(cohort_pivot.index):
    for col_idx, col_label in enumerate(cohort_pivot.columns):
        value = cohort_pivot.loc[row_label, col_label]
        if pd.notna(value):
            ax.text(col_idx, row_idx, f'{value:.0%}', ha='center', va='center', color='black', fontsize=8)

plt.colorbar(heatmap, ax=ax, fraction=0.046, pad=0.04)
plt.tight_layout()
plt.show()

**Observation**

The cohort view makes retention easier to compare across conversion months. For a recruiter, this is often one of the strongest visuals because it connects acquisition timing with downstream subscription quality.

## 11. Risk Segments and First Behavioral Signals

In [ ]:
risk_summary = (
    paid_customers.groupby(['acquisition_channel', 'plan_type', 'usage_segment'])['is_cancelled']
    .agg(['mean', 'count'])
    .reset_index()
    .rename(columns={'mean': 'churn_share', 'count': 'customers'})
    .sort_values(['churn_share', 'customers'], ascending=[False, False])
)

risk_summary.head(10)

In [ ]:
behavior_signals = (
    paid_customers.groupby('is_cancelled')
    .agg(
        customers=('customer_id', 'nunique'),
        avg_observed_months=('observed_months', 'mean'),
        avg_app_opens=('avg_app_opens', 'mean'),
        avg_transactions=('avg_transactions', 'mean'),
        avg_dashboard_views=('avg_dashboard_views', 'mean'),
    )
    .rename(index={False: 'Retained', True: 'Churned'})
)
behavior_signals

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

top_risk = risk_summary.head(8).iloc[::-1]
labels = top_risk.apply(lambda row: f"{row['acquisition_channel']} | {row['plan_type']} | {row['usage_segment']}", axis=1)
axes[0].barh(labels, top_risk['churn_share'], color='#dc2626')
axes[0].set_title('Highest Risk Segments')
axes[0].set_xlabel('Churn share')

behavior_plot = behavior_signals[['avg_app_opens', 'avg_transactions', 'avg_dashboard_views']]
behavior_plot.T.plot(kind='bar', ax=axes[1], color=['#16a34a', '#f97316'])
axes[1].set_title('Observed Behavior: Retained vs Churned')
axes[1].set_xlabel('Metric')
axes[1].set_ylabel('Average')

plt.tight_layout()
plt.show()

**Observation**

The risk view helps identify where churn is concentrated, while the retained vs churned comparison gives first behavioral signals. In this simulated dataset, churned customers typically show lower observed engagement and weaker product usage intensity.

## 12. Final Business Takeaways

In [ ]:
top_risk_row = risk_summary.iloc[0]
best_channel = churn_by_channel['churn_share'].idxmin()
worst_channel = churn_by_channel['churn_share'].idxmax()

print('Main churn and retention takeaways')
print(f'- Global churn share among paid customers is {global_churn_share:.1%}.')
print(f'- Monthly plans churn materially more than annual plans ({churn_by_plan.loc["monthly", "churn_share"]:.1%} vs {churn_by_plan.loc["annual", "churn_share"]:.1%}).')
print(f'- {worst_channel} appears riskier on churn, while {best_channel} retains better among converted customers.')
print(f'- Activated customers retain better than non-activated customers after conversion.')
print('- Lower observed usage is strongly associated with higher churn risk.')
print('- Paid retention declines over time in a way that is consistent with an early-stage B2C subscription business.')
print(f'- One of the riskiest segment combinations is {top_risk_row["acquisition_channel"]} | {top_risk_row["plan_type"]} | {top_risk_row["usage_segment"]}.')

print('\nPossible business actions')
print('- Strengthen onboarding flows that drive activation, especially first budget creation and first transaction import.')
print('- Prioritize Organic and Referral quality loops, since they appear stronger on both conversion and retention.')
print('- Build retention nudges for low-usage monthly subscribers before they reach inactivity.')
print('- Promote annual plans to the most engaged users, especially medium and power segments.')
print('- Monitor Paid Social quality more closely and review whether its acquisition efficiency compensates for weaker retention.')